
# TRUST4 — универсальная реконструкция BCR из синтетических PE150

Этот notebook запускает **TRUST4** на всех актуальных ветках симуляции человека и мыши,
а затем приводит нативные результаты TRUST4 к единому формату для последующего benchmark.

Поддерживаемая матрица:

| key | species | error model | fragmentation |
|---|---|---|---|
| `human_novaseq_random_cut` | human | NovaSeq | uniform single random-cut |
| `human_custom_random_cut` | human | custom UMI-consensus | uniform single random-cut |
| `human_novaseq_ultrasonic` | human | NovaSeq | ultrasonic-like |
| `human_custom_ultrasonic` | human | custom UMI-consensus | ultrasonic-like |
| `mouse_novaseq_random_cut` | mouse | NovaSeq | uniform single random-cut |
| `mouse_novaseq_ultrasonic` | mouse | NovaSeq | ultrasonic-like |

## Почему TRUST4 запускается именно так

TRUST4 состоит из трёх основных стадий: candidate-read extraction → de novo assembly →
annotation. Для raw paired FASTQ координаты генов в геноме для `-f` не обязательны:
официальная документация допускает использовать один и тот же IMGT V/D/J/C FASTA
как `-f` и `--ref`.

Наши simulated FASTQ уже состоят только из BCR-фрагментов. Поэтому основной benchmark-режим:

- `--noExtraction` — передаём все синтетические BCR reads непосредственно в assembly;
- **не используем `--repseq` по умолчанию**;
- mate-pair extension сохраняется;
- `--outputReadAssignment` включён для будущей truth-validation;
- `--clean 0` сохраняет нативные intermediate/output-файлы.

Причина не использовать `--repseq` автоматически: в текущем `run-trust4` этот флаг
преобразуется в `--trimLevel 2 --skipMateExtension`. Для наших фрагментированных PE150
данных mate-pair information является полезной частью задачи восстановления.

Дополнительно можно переключить `TRUST4_INPUT_MODE` на `native_extraction`, чтобы отдельно
оценить полный стандартный pipeline TRUST4 с candidate-read extraction.

## Зафиксированная версия

Notebook ориентирован на upstream `liulab-dfci/TRUST4`:

- commit: `4032f90b82b3cca6d56aaa7645cc7ee51dd3889c`
- `run-trust4`: `v1.1.10-r639`

Источники:
- upstream repository: `liulab-dfci/TRUST4`
- Song et al. *Nature Methods* 2021, DOI: `10.1038/s41592-021-01142-2`

Notebook не выполняет финальное сравнение с truth. Его задача — получить реконструированные
contigs и единый нормализованный output, который дальше можно сравнивать с исходными V–J templates.


## 1. Окружение и расположение проекта

In [ ]:

import os
import sys
import csv
import re
import json
import gzip
import time
import shutil
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

def locate_project_root():
    candidates = []
    if os.environ.get("BCR_VOLUME"):
        candidates.append(Path(os.environ["BCR_VOLUME"]).expanduser().resolve())
    candidates.extend([
        Path("/data/user/epishkin"),
        Path.cwd().resolve(),
        Path("/Users/epishkin/workspace/bcr-assembler"),
    ])

    seen = set()
    for start in candidates:
        for candidate in (start, *start.parents):
            candidate = candidate.resolve()
            if candidate in seen:
                continue
            seen.add(candidate)
            if (
                (candidate / "results").is_dir()
                and (
                    (candidate / "results/PRJEB30386").exists()
                    or (candidate / "results/ERP003950_fastp_q30_u40").exists()
                )
            ):
                return candidate
    raise FileNotFoundError(
        "Cannot locate bcr-assembler project root. "
        "Set BCR_VOLUME to the directory containing results/."
    )

VOLUME = locate_project_root()
print("Project root:", VOLUME)

def which_or_none(name):
    return shutil.which(name)

for tool in ("git", "make", "perl", "wget"):
    print(f"{tool}: {which_or_none(tool)}")


## 2. Реестр simulation branches и параметры запуска

In [ ]:

SIMULATIONS = {
    "human_novaseq_random_cut": {
        "species": "human",
        "dataset": "PRJEB30386",
        "branch": "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut",
        "fragmentation": "random_cut",
        "error_model": "novaseq",
        "expected_samples": ["PRJEB30386_all_chains"],
    },
    "human_custom_random_cut": {
        "species": "human",
        "dataset": "PRJEB30386",
        "branch": "insilicoseq_150bp_custom_umi_consensus_post_annotation_filtered_random_cut",
        "fragmentation": "random_cut",
        "error_model": "custom_umi_consensus",
        "expected_samples": ["PRJEB30386_all_chains"],
    },
    "human_novaseq_ultrasonic": {
        "species": "human",
        "dataset": "PRJEB30386",
        "branch": "insilicoseq_150bp_novaseq_post_annotation_filtered_ultrasonic_like",
        "fragmentation": "ultrasonic_like",
        "error_model": "novaseq",
        "expected_samples": ["PRJEB30386_all_chains"],
    },
    "human_custom_ultrasonic": {
        "species": "human",
        "dataset": "PRJEB30386",
        "branch": "insilicoseq_150bp_custom_umi_consensus_post_annotation_filtered_ultrasonic_like",
        "fragmentation": "ultrasonic_like",
        "error_model": "custom_umi_consensus",
        "expected_samples": ["PRJEB30386_all_chains"],
    },
    "mouse_novaseq_random_cut": {
        "species": "mouse",
        "dataset": "ERP003950_fastp_q30_u40",
        "branch": "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut",
        "fragmentation": "random_cut",
        "error_model": "novaseq",
        "expected_samples": [
            "ERR346596", "ERR346597", "ERR346598",
            "ERR346599", "ERR346600", "ERR346601",
        ],
    },
    "mouse_novaseq_ultrasonic": {
        "species": "mouse",
        "dataset": "ERP003950_fastp_q30_u40",
        "branch": "insilicoseq_150bp_novaseq_post_annotation_filtered_ultrasonic_like",
        "fragmentation": "ultrasonic_like",
        "error_model": "novaseq",
        "expected_samples": [
            "ERR346596", "ERR346597", "ERR346598",
            "ERR346599", "ERR346600", "ERR346601",
        ],
    },
}

# ------------------------------------------------------------------
# ОСНОВНАЯ НАСТРОЙКА
# ------------------------------------------------------------------
# Можно указать один или несколько ключей.
# Специальные значения:
#   RUN_SELECTION = "available"  -> все ветки, для которых уже есть 06_fastq_pe150
#   RUN_SELECTION = "all"        -> требовать наличие всех шести веток
RUN_SELECTION = ["human_novaseq_random_cut"]

NPROC = 8
FORCE = False
REQUIRE_SIMULATION_QC = True

# Главный benchmark-режим:
#   pure_bcr_no_extraction -> все synthetic BCR reads сразу в assembly
#   native_extraction      -> стандартная candidate-read extraction TRUST4
TRUST4_INPUT_MODE = "pure_bcr_no_extraction"

OUTPUT_READ_ASSIGNMENT = True
CLEAN_LEVEL = 0
EXTRA_TRUST4_ARGS = []

# `--repseq` намеренно выключен: он включает --skipMateExtension.
USE_REPSEQ_FLAG = False

# Воспроизводимость TRUST4.
TRUST4_REPO = "https://github.com/liulab-dfci/TRUST4.git"
TRUST4_PIN = "4032f90b82b3cca6d56aaa7645cc7ee51dd3889c"
TRUST4_EXPECTED_VERSION = "v1.1.10-r639"
AUTO_INSTALL_TRUST4 = True
AUTO_BUILD_IMGT_REFERENCE = True

TOOLS_DIR = VOLUME / "tools"
TRUST4_DIR = TOOLS_DIR / f"TRUST4_{TRUST4_PIN[:12]}"
REF_DIR = VOLUME / "seq_refs" / "trust4"

IMGT_SPECIES = {
    "human": "Homo_sapiens",
    "mouse": "Mus_musculus",
}

MODE_DIRNAME = {
    "pure_bcr_no_extraction": "no_extraction_mate_extension",
    "native_extraction": "native_extraction_mate_extension",
}[TRUST4_INPUT_MODE]

print("TRUST4 mode:", TRUST4_INPUT_MODE)
print("Output subdir:", MODE_DIRNAME)


## 3. Общие функции

In [ ]:

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            chunk = fh.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n")
    tmp.replace(path)

def write_tsv(path, rows, fields):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=fields, delimiter="\t", extrasaction="ignore")
        w.writeheader()
        for row in rows:
            w.writerow(row)
    tmp.replace(path)

def run_checked(cmd, cwd=None, stdout_path=None):
    print("RUN:", " ".join(map(str, cmd)))
    if stdout_path is None:
        subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)
        return
    stdout_path = Path(stdout_path)
    stdout_path.parent.mkdir(parents=True, exist_ok=True)
    with open(stdout_path, "wb") as out:
        subprocess.run(list(map(str, cmd)), cwd=cwd, stdout=out, check=True)

def run_logged_with_heartbeat(cmd, log_path, cwd=None, heartbeat_seconds=60):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("RUN:", " ".join(map(str, cmd)))
    print("LOG:", log_path)

    with open(log_path, "w") as log:
        proc = subprocess.Popen(
            list(map(str, cmd)),
            cwd=cwd,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
        last = time.time()
        while proc.poll() is None:
            time.sleep(2)
            if time.time() - last >= heartbeat_seconds:
                print(f"[{time.strftime('%H:%M:%S')}] TRUST4 still running...")
                last = time.time()

    if proc.returncode != 0:
        tail = ""
        try:
            tail = "\n".join(log_path.read_text(errors="replace").splitlines()[-80:])
        except Exception:
            pass
        raise RuntimeError(
            f"TRUST4 failed with exit code {proc.returncode}.\n"
            f"Last log lines:\n{tail}"
        )

def iter_fasta(path):
    name = None
    seq = []
    with open(path) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(seq)
                name = line[1:]
                seq = []
            else:
                seq.append(line.strip())
        if name is not None:
            yield name, "".join(seq)

def count_noncomment_rows(path):
    n = 0
    with open(path) as fh:
        for line in fh:
            if line.strip() and not line.startswith("#"):
                n += 1
    return n

def count_airr_rows(path):
    with open(path) as fh:
        n = sum(1 for line in fh if line.strip())
    return max(0, n - 1)

def infer_locus(*genes):
    for gene in genes:
        if not gene or gene in ("*", "."):
            continue
        gene = gene.split(",")[0]
        for locus in ("IGH", "IGK", "IGL"):
            if gene.startswith(locus):
                return locus
    return "unknown"


## 4. Выбор существующих simulation outputs и preflight

In [ ]:

def simulation_paths(spec):
    dataset_dir = VOLUME / "results" / spec["dataset"]
    sim_dir = dataset_dir / "simulated" / spec["branch"]
    return {
        "dataset_dir": dataset_dir,
        "sim_dir": sim_dir,
        "fastq_dir": sim_dir / "06_fastq_pe150",
        "qc_file": sim_dir / "qc" / "final_qc.tsv",
        "trust4_root": sim_dir / "trust4" / MODE_DIRNAME,
    }

def branch_available(spec):
    p = simulation_paths(spec)
    return p["fastq_dir"].is_dir() and any(p["fastq_dir"].glob("*_R1.fastq.gz"))

def resolve_selection():
    if RUN_SELECTION == "all":
        keys = list(SIMULATIONS)
    elif RUN_SELECTION == "available":
        keys = [k for k, spec in SIMULATIONS.items() if branch_available(spec)]
    elif isinstance(RUN_SELECTION, (list, tuple)):
        keys = list(RUN_SELECTION)
    else:
        raise TypeError("RUN_SELECTION must be a list, 'available', or 'all'")

    unknown = [k for k in keys if k not in SIMULATIONS]
    if unknown:
        raise KeyError(f"Unknown simulation keys: {unknown}")

    if not keys:
        raise RuntimeError("No simulation branches selected/found.")
    return keys

def read_final_qc(qc_path):
    if not qc_path.exists():
        return {}
    rows = {}
    with open(qc_path) as fh:
        for row in csv.DictReader(fh, delimiter="\t"):
            rows[row["sample"]] = row
    return rows

def discover_pairs(spec):
    p = simulation_paths(spec)
    fastq_dir = p["fastq_dir"]
    if not fastq_dir.is_dir():
        raise FileNotFoundError(f"Missing simulation FASTQ directory: {fastq_dir}")

    pairs = {}
    for r1 in sorted(fastq_dir.glob("*_R1.fastq.gz")):
        sample = r1.name[:-len("_R1.fastq.gz")]
        r2 = fastq_dir / f"{sample}_R2.fastq.gz"
        if not r2.exists():
            raise FileNotFoundError(f"Missing R2 for {sample}: {r2}")
        pairs[sample] = (r1, r2)

    expected = set(spec["expected_samples"])
    observed = set(pairs)
    if observed != expected:
        raise RuntimeError(
            f"{spec['branch']}: sample set mismatch.\n"
            f"expected={sorted(expected)}\nobserved={sorted(observed)}"
        )

    if REQUIRE_SIMULATION_QC:
        qc = read_final_qc(p["qc_file"])
        if set(qc) != expected:
            raise RuntimeError(
                f"Simulation final_qc.tsv missing/incomplete for {spec['branch']}: {p['qc_file']}"
            )
        bad = [s for s in expected if str(qc[s].get("valid", "")).lower() not in ("true", "1")]
        if bad:
            raise RuntimeError(f"Simulation QC failed for: {bad}")

    return pairs

SELECTED_KEYS = resolve_selection()
RUN_PLAN = []

for key in SELECTED_KEYS:
    spec = SIMULATIONS[key]
    pairs = discover_pairs(spec)
    p = simulation_paths(spec)
    for sample, (r1, r2) in pairs.items():
        RUN_PLAN.append({
            "simulation_key": key,
            "species": spec["species"],
            "dataset": spec["dataset"],
            "branch": spec["branch"],
            "fragmentation": spec["fragmentation"],
            "error_model": spec["error_model"],
            "sample": sample,
            "r1": r1,
            "r2": r2,
            "trust4_root": p["trust4_root"],
        })

print("Selected branches:", SELECTED_KEYS)
print("TRUST4 jobs:", len(RUN_PLAN))
for job in RUN_PLAN:
    print(
        f"- {job['simulation_key']} | {job['sample']} | "
        f"{job['r1'].name} + {job['r2'].name}"
    )



## 5. Установка TRUST4 и IMGT reference

Для воспроизводимости notebook использует отдельный checkout, привязанный к конкретному commit.

Для raw FASTQ TRUST4 documentation разрешает использовать IMGT reference одновременно как:

```text
-f species_IMGT+C.fa
--ref species_IMGT+C.fa
```

Поэтому отдельный genome-coordinate `hg38_bcrtcr.fa/mm10_bcrtcr.fa` здесь не нужен.

`BuildImgtAnnot.pl` в текущем upstream загружает общий IMGT GENE-DB FASTA и фильтрует его
по виду. Для человека используется строка `Homo_sapiens`, для мыши `Mus_musculus`.
Получившийся FASTA кэшируется в `seq_refs/trust4/`, а SHA256 записывается в provenance.


In [ ]:

def ensure_trust4():
    TOOLS_DIR.mkdir(parents=True, exist_ok=True)

    if not TRUST4_DIR.exists():
        if not AUTO_INSTALL_TRUST4:
            raise FileNotFoundError(
                f"TRUST4 checkout not found: {TRUST4_DIR}. "
                "Set AUTO_INSTALL_TRUST4=True or install it manually."
            )
        for tool in ("git", "make"):
            if not shutil.which(tool):
                raise RuntimeError(f"Required build tool not found: {tool}")

        run_checked(["git", "clone", TRUST4_REPO, str(TRUST4_DIR)])
        run_checked(["git", "checkout", "--detach", TRUST4_PIN], cwd=TRUST4_DIR)
    else:
        git_dir = TRUST4_DIR / ".git"
        if not git_dir.exists():
            raise RuntimeError(f"{TRUST4_DIR} exists but is not a git checkout")

    head = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=TRUST4_DIR, text=True
    ).strip()
    if head != TRUST4_PIN:
        raise RuntimeError(
            f"TRUST4 checkout is at {head}, expected pinned {TRUST4_PIN}. "
            f"Use a clean checkout at {TRUST4_DIR}."
        )

    run_script = TRUST4_DIR / "run-trust4"
    binary = TRUST4_DIR / "trust4"
    if not binary.exists():
        run_checked(["make", "-j", str(max(1, min(NPROC, 8)))], cwd=TRUST4_DIR)

    probe = subprocess.run(
        [str(run_script)],
        cwd=TRUST4_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    m = re.search(r"TRUST4\s+(v[^\s]+)\s+usage", probe.stdout)
    version = m.group(1) if m else "unknown"
    if version != TRUST4_EXPECTED_VERSION:
        raise RuntimeError(
            f"Unexpected TRUST4 version: {version}; expected {TRUST4_EXPECTED_VERSION}"
        )

    print("TRUST4:", run_script)
    print("commit:", head)
    print("version:", version)
    return run_script, head, version

def validate_imgt_reference(path, species):
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"Empty/missing IMGT reference: {path}")

    headers = []
    with open(path) as fh:
        for line in fh:
            if line.startswith(">"):
                headers.append(line[1:].strip())

    if not headers:
        raise RuntimeError(f"No FASTA records in {path}")

    loci = {
        locus: sum(h.startswith(locus) for h in headers)
        for locus in ("IGH", "IGK", "IGL")
    }
    if loci["IGH"] == 0:
        raise RuntimeError(f"{species}: IMGT reference contains no IGH genes")

    return {"records": len(headers), "loci": loci}

def ensure_imgt_reference(species, trust4_dir, trust4_commit):
    REF_DIR.mkdir(parents=True, exist_ok=True)
    ref_path = REF_DIR / f"{species}_IMGT+C.fa"
    manifest_path = REF_DIR / f"{species}_IMGT+C.manifest.json"

    if not ref_path.exists():
        if not AUTO_BUILD_IMGT_REFERENCE:
            raise FileNotFoundError(
                f"Reference not found: {ref_path}. "
                "Set AUTO_BUILD_IMGT_REFERENCE=True or place a prepared IMGT+C FASTA there."
            )
        for tool in ("perl", "wget"):
            if not shutil.which(tool):
                raise RuntimeError(f"Required reference-build tool not found: {tool}")

        species_name = IMGT_SPECIES[species]
        tmp = ref_path.with_suffix(".fa.tmp")
        log = REF_DIR / f"{species}_BuildImgtAnnot.log"

        print(f"Building IMGT reference for {species} ({species_name})...")
        with open(tmp, "wb") as out, open(log, "wb") as err:
            proc = subprocess.run(
                ["perl", str(trust4_dir / "BuildImgtAnnot.pl"), species_name],
                cwd=REF_DIR,
                stdout=out,
                stderr=err,
            )
        if proc.returncode != 0:
            tmp.unlink(missing_ok=True)
            raise RuntimeError(
                f"BuildImgtAnnot.pl failed for {species}. See {log}"
            )
        tmp.replace(ref_path)

    stats = validate_imgt_reference(ref_path, species)
    digest = sha256_file(ref_path)

    manifest = {
        "species": species,
        "imgt_species_token": IMGT_SPECIES[species],
        "reference_path": str(ref_path),
        "sha256": digest,
        "records": stats["records"],
        "loci": stats["loci"],
        "trust4_commit_used_for_build_script": trust4_commit,
        "checked_at_utc": utc_now(),
    }
    atomic_write_json(manifest_path, manifest)

    print(
        f"{species} reference: {ref_path} | "
        f"records={stats['records']} | loci={stats['loci']} | sha256={digest[:12]}..."
    )
    return ref_path, manifest


## 6. Построение команды TRUST4 и безопасный rerun

In [ ]:

def file_identity(path):
    st = path.stat()
    return {
        "path": str(path.resolve()),
        "size": int(st.st_size),
        "mtime_ns": int(st.st_mtime_ns),
    }

def build_trust4_command(run_script, ref_path, job, sample_out):
    prefix_name = f"TRUST4_{job['sample']}"

    cmd = [
        str(run_script),
        "-f", str(ref_path),
        "--ref", str(ref_path),
        "-1", str(job["r1"]),
        "-2", str(job["r2"]),
        "-o", prefix_name,
        "--od", str(sample_out),
        "-t", str(NPROC),
    ]

    if TRUST4_INPUT_MODE == "pure_bcr_no_extraction":
        cmd.append("--noExtraction")
    elif TRUST4_INPUT_MODE == "native_extraction":
        pass
    else:
        raise ValueError(TRUST4_INPUT_MODE)

    if OUTPUT_READ_ASSIGNMENT:
        cmd.append("--outputReadAssignment")

    if USE_REPSEQ_FLAG:
        cmd.append("--repseq")

    cmd.extend(["--clean", str(CLEAN_LEVEL)])
    cmd.extend(map(str, EXTRA_TRUST4_ARGS))
    return cmd, prefix_name

def expected_output_paths(sample_out, prefix_name):
    prefix = sample_out / prefix_name
    out = {
        "annot_fa": Path(str(prefix) + "_annot.fa"),
        "cdr3": Path(str(prefix) + "_cdr3.out"),
        "report": Path(str(prefix) + "_report.tsv"),
        "airr": Path(str(prefix) + "_airr.tsv"),
    }
    if OUTPUT_READ_ASSIGNMENT:
        out["assign"] = Path(str(prefix) + "_assign.out")
    return out

def signature_payload(job, cmd, ref_manifest, trust4_commit, trust4_version):
    payload = {
        "simulation_key": job["simulation_key"],
        "species": job["species"],
        "dataset": job["dataset"],
        "branch": job["branch"],
        "fragmentation": job["fragmentation"],
        "error_model": job["error_model"],
        "sample": job["sample"],
        "r1": file_identity(job["r1"]),
        "r2": file_identity(job["r2"]),
        "reference_sha256": ref_manifest["sha256"],
        "trust4_commit": trust4_commit,
        "trust4_version": trust4_version,
        "trust4_input_mode": TRUST4_INPUT_MODE,
        "use_repseq_flag": USE_REPSEQ_FLAG,
        "output_read_assignment": OUTPUT_READ_ASSIGNMENT,
        "clean_level": CLEAN_LEVEL,
        "command": list(map(str, cmd)),
    }
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    payload["signature_sha256"] = hashlib.sha256(canonical.encode()).hexdigest()
    return payload

def run_one_job(run_script, ref_path, ref_manifest, trust4_commit, trust4_version, job):
    sample_out = Path(job["trust4_root"]) / job["sample"]
    manifest_path = sample_out / "run_manifest.json"

    cmd, prefix_name = build_trust4_command(run_script, ref_path, job, sample_out)
    sig = signature_payload(job, cmd, ref_manifest, trust4_commit, trust4_version)
    expected = expected_output_paths(sample_out, prefix_name)

    complete = all(p.exists() and p.stat().st_size > 0 for p in expected.values())

    if manifest_path.exists() and complete and not FORCE:
        old = json.loads(manifest_path.read_text())
        if old.get("signature_sha256") == sig["signature_sha256"]:
            print(f"[skip identical] {job['simulation_key']} / {job['sample']}")
            return expected, old
        raise RuntimeError(
            f"Existing TRUST4 output has different configuration: {sample_out}\n"
            "Set FORCE=True to replace this sample output, or change output mode/config."
        )

    if sample_out.exists() and FORCE:
        shutil.rmtree(sample_out)

    if sample_out.exists() and any(sample_out.iterdir()) and not FORCE:
        raise RuntimeError(
            f"Non-empty output directory without matching complete manifest: {sample_out}. "
            "Inspect it or set FORCE=True."
        )

    sample_out.mkdir(parents=True, exist_ok=True)
    sig["started_at_utc"] = utc_now()
    atomic_write_json(manifest_path, sig)

    log_path = sample_out / "trust4.log"
    run_logged_with_heartbeat(cmd, log_path, cwd=TRUST4_DIR)

    missing = [str(p) for p in expected.values() if not (p.exists() and p.stat().st_size > 0)]
    if missing:
        raise RuntimeError(f"TRUST4 finished but expected outputs are missing/empty: {missing}")

    sig["completed_at_utc"] = utc_now()
    sig["status"] = "complete"
    sig["outputs"] = {k: str(v) for k, v in expected.items()}
    atomic_write_json(manifest_path, sig)

    print(f"[complete] {job['simulation_key']} / {job['sample']}")
    return expected, sig



## 7. Нормализация TRUST4 outputs

Нативные TRUST4 outputs сохраняются без изменений.

Дополнительно для каждой simulation branch создаются:

- `normalized/reconstructed_contigs.fasta`
- `normalized/reconstructed_contigs.tsv`
- `normalized/cdr3_normalized.tsv`
- `normalized/trust4_run_summary.tsv`
- `normalized/trust4_provenance.json`

`reconstructed_contigs.fasta` — основной sequence-level output для последующего сравнения
с simulation truth.

`cdr3_normalized.tsv` следует структуре `trust_cdr3.out`, документированной TRUST4:

`consensus_id, index_within_consensus, V, D, J, C, CDR1, CDR2, CDR3,
CDR3_score, read_fragment_count, CDR3_germline_similarity, complete_vdj_assembly`.


In [ ]:

CDR3_FIELDS = [
    "consensus_id",
    "index_within_consensus",
    "V_gene",
    "D_gene",
    "J_gene",
    "C_gene",
    "CDR1",
    "CDR2",
    "CDR3",
    "CDR3_score",
    "read_fragment_count",
    "CDR3_germline_similarity",
    "complete_vdj_assembly",
]

def parse_cdr3_file(path, sample, simulation_key):
    rows = []
    with open(path) as fh:
        for lineno, line in enumerate(fh, 1):
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            cols = line.split("\t")
            if len(cols) != len(CDR3_FIELDS):
                cols = line.split()
            if len(cols) != len(CDR3_FIELDS):
                raise ValueError(
                    f"Unexpected trust_cdr3.out format at {path}:{lineno}: "
                    f"{len(cols)} columns"
                )
            row = dict(zip(CDR3_FIELDS, cols))
            row["sample"] = sample
            row["simulation_key"] = simulation_key
            row["locus"] = infer_locus(
                row["V_gene"], row["J_gene"], row["C_gene"]
            )
            rows.append(row)
    return rows

def parse_contigs(path, sample, simulation_key):
    rows = []
    for raw_header, seq in iter_fasta(path):
        consensus_id = raw_header.split()[0]
        rows.append({
            "simulation_key": simulation_key,
            "sample": sample,
            "consensus_id": consensus_id,
            "sequence_length": len(seq),
            "raw_header": raw_header,
            "sequence": seq,
        })
    return rows

def normalize_branch(key, jobs_and_outputs, trust4_commit, trust4_version, ref_manifest):
    spec = SIMULATIONS[key]
    p = simulation_paths(spec)
    norm_dir = p["trust4_root"] / "normalized"
    norm_dir.mkdir(parents=True, exist_ok=True)

    contigs = []
    cdr3_rows = []
    summary = []

    for job, outputs, manifest in jobs_and_outputs:
        sample = job["sample"]
        sample_contigs = parse_contigs(outputs["annot_fa"], sample, key)
        sample_cdr3 = parse_cdr3_file(outputs["cdr3"], sample, key)

        contigs.extend(sample_contigs)
        cdr3_rows.extend(sample_cdr3)

        complete_vdj = sum(
            str(r["complete_vdj_assembly"]).lower() in ("1", "true")
            for r in sample_cdr3
        )

        locus_counts = {
            locus: sum(r["locus"] == locus for r in sample_cdr3)
            for locus in ("IGH", "IGK", "IGL")
        }

        summary.append({
            "simulation_key": key,
            "species": spec["species"],
            "dataset": spec["dataset"],
            "branch": spec["branch"],
            "fragmentation": spec["fragmentation"],
            "error_model": spec["error_model"],
            "trust4_input_mode": TRUST4_INPUT_MODE,
            "sample": sample,
            "contigs": len(sample_contigs),
            "cdr3_rows": len(sample_cdr3),
            "complete_vdj_rows": complete_vdj,
            "IGH_cdr3_rows": locus_counts["IGH"],
            "IGK_cdr3_rows": locus_counts["IGK"],
            "IGL_cdr3_rows": locus_counts["IGL"],
            "report_rows": count_noncomment_rows(outputs["report"]),
            "airr_rows": count_airr_rows(outputs["airr"]),
            "trust4_output_dir": str(Path(job["trust4_root"]) / sample),
        })

    fasta_path = norm_dir / "reconstructed_contigs.fasta"
    with open(fasta_path.with_suffix(".fasta.tmp"), "w") as fh:
        for row in contigs:
            fh.write(f">{row['sample']}|{row['consensus_id']}\n{row['sequence']}\n")
    fasta_path.with_suffix(".fasta.tmp").replace(fasta_path)

    write_tsv(
        norm_dir / "reconstructed_contigs.tsv",
        contigs,
        [
            "simulation_key", "sample", "consensus_id",
            "sequence_length", "raw_header", "sequence",
        ],
    )

    cdr3_out_fields = ["simulation_key", "sample", "locus"] + CDR3_FIELDS
    write_tsv(
        norm_dir / "cdr3_normalized.tsv",
        cdr3_rows,
        cdr3_out_fields,
    )

    summary_fields = [
        "simulation_key", "species", "dataset", "branch",
        "fragmentation", "error_model", "trust4_input_mode",
        "sample", "contigs", "cdr3_rows", "complete_vdj_rows",
        "IGH_cdr3_rows", "IGK_cdr3_rows", "IGL_cdr3_rows",
        "report_rows", "airr_rows", "trust4_output_dir",
    ]
    write_tsv(norm_dir / "trust4_run_summary.tsv", summary, summary_fields)

    provenance = {
        "generated_at_utc": utc_now(),
        "simulation": spec,
        "trust4": {
            "repository": "liulab-dfci/TRUST4",
            "commit": trust4_commit,
            "version": trust4_version,
            "input_mode": TRUST4_INPUT_MODE,
            "repseq_flag": USE_REPSEQ_FLAG,
            "output_read_assignment": OUTPUT_READ_ASSIGNMENT,
            "clean_level": CLEAN_LEVEL,
        },
        "reference": ref_manifest,
        "normalized_outputs": {
            "reconstructed_contigs_fasta": str(fasta_path),
            "reconstructed_contigs_tsv": str(norm_dir / "reconstructed_contigs.tsv"),
            "cdr3_tsv": str(norm_dir / "cdr3_normalized.tsv"),
            "summary_tsv": str(norm_dir / "trust4_run_summary.tsv"),
        },
    }
    atomic_write_json(norm_dir / "trust4_provenance.json", provenance)

    print(f"\nNormalized: {key}")
    print("  contigs:", len(contigs))
    print("  cdr3 rows:", len(cdr3_rows))
    print("  dir:", norm_dir)
    return summary


## 8. Запуск TRUST4

In [ ]:

RUN_TRUST4, TRUST4_COMMIT, TRUST4_VERSION = ensure_trust4()

species_needed = sorted({SIMULATIONS[k]["species"] for k in SELECTED_KEYS})
REFERENCES = {}
for species in species_needed:
    REFERENCES[species] = ensure_imgt_reference(
        species,
        TRUST4_DIR,
        TRUST4_COMMIT,
    )

RESULTS_BY_BRANCH = {key: [] for key in SELECTED_KEYS}

for job in RUN_PLAN:
    ref_path, ref_manifest = REFERENCES[job["species"]]
    outputs, manifest = run_one_job(
        RUN_TRUST4,
        ref_path,
        ref_manifest,
        TRUST4_COMMIT,
        TRUST4_VERSION,
        job,
    )
    RESULTS_BY_BRANCH[job["simulation_key"]].append(
        (job, outputs, manifest)
    )

ALL_SUMMARY = []
for key in SELECTED_KEYS:
    species = SIMULATIONS[key]["species"]
    ref_path, ref_manifest = REFERENCES[species]
    branch_summary = normalize_branch(
        key,
        RESULTS_BY_BRANCH[key],
        TRUST4_COMMIT,
        TRUST4_VERSION,
        ref_manifest,
    )
    ALL_SUMMARY.extend(branch_summary)

print("\n=== TRUST4 reconstruction complete ===")
for row in ALL_SUMMARY:
    print(
        f"{row['simulation_key']} | {row['sample']} | "
        f"contigs={row['contigs']} | cdr3={row['cdr3_rows']} | "
        f"complete_vdj={row['complete_vdj_rows']}"
    )



## 9. Как запускать разные ветки

Одна ветка:

```python
RUN_SELECTION = ["human_novaseq_random_cut"]
```

Несколько:

```python
RUN_SELECTION = [
    "human_novaseq_random_cut",
    "human_novaseq_ultrasonic",
]
```

Все simulation branches, которые уже существуют на диске:

```python
RUN_SELECTION = "available"
```

Требовать и запускать все шесть:

```python
RUN_SELECTION = "all"
```

### Стандартный benchmark-режим

```python
TRUST4_INPUT_MODE = "pure_bcr_no_extraction"
USE_REPSEQ_FLAG = False
```

Так TRUST4 получает все synthetic BCR reads непосредственно в assembly и сохраняет
paired-end mate extension.

### Контрольный end-to-end режим TRUST4

```python
TRUST4_INPUT_MODE = "native_extraction"
USE_REPSEQ_FLAG = False
```

В этом случае TRUST4 сначала выполняет собственный candidate-read extraction.
Эти два режима **не следует смешивать в одной метрике**: первый измеряет главным образом
способность assembly/reconstruction, второй — полный pipeline extraction + assembly + annotation.



## 10. Что будет сравниваться с truth дальше

Этот notebook намеренно не вычисляет recall/precision восстановления.

Следующий benchmark notebook должен сравнить:

1. `trust4/.../normalized/reconstructed_contigs.fasta`
2. simulation truth:
   `00_primary_truth/*_templates.fasta`

Минимальный набор метрик:

- exact full V–J reconstruction;
- near-exact identity;
- recovered fraction по исходным templates;
- partial reconstruction;
- split / merge / chimera;
- locus correctness (IGH / IGK / IGL);
- зависимость восстановления от fragment coverage/reconstructability;
- отдельно random-cut vs ultrasonic-like;
- отдельно NovaSeq vs custom error model.

Это сохраняет чёткое разделение:
**simulation → assembler → truth benchmark**.
